# Climate integration by SSP × GCM × time period (AGEB level)

Stacks the 240 `data.csv` files in `data/hazard/ageb/climate_by_ssp_gcm/` (4 SSP × 15 GCM × 4 time periods)
into a single long table: **one row per AGEB × ssp × gcm × time_period**.

Folder structure: `ssp_XXX/gcm_XXX/time_period_YYYY_YYYY/data.csv`.
Each CSV holds `CVEGEO` + hazard indicators (`heat_*`, `drought_*`, `cold_*`, `flood_*`);
the `ssp`, `gcm` and `time_period` columns are derived from the file path.

Output: `data/hazard/ageb/climate_ssp_gcm_all_ageb.parquet` (the equivalent CSV would be ~3 GB and Excel cannot open it; `CVEGEO` is stored as text to keep leading zeros).

In [1]:
from pathlib import Path
import re
import pandas as pd

# The data/ folder is in .gitignore and lives in the main checkout;
# search upward from the current directory (also works from a worktree).
def find_data_dir(start=Path.cwd().resolve()):
    for p in [start, *start.parents]:
        d = p / "data" / "hazard" / "ageb" / "climate_by_ssp_gcm"
        if d.is_dir():
            return d
    raise FileNotFoundError("data/hazard/ageb/climate_by_ssp_gcm not found")

IN_DIR = find_data_dir()
OUT_DIR = IN_DIR.parent
OUT_PARQUET = OUT_DIR / "climate_ssp_gcm_all_ageb.parquet"
print(IN_DIR)

/Users/fabianfuentes/git/adatpation_ageb_data/data/hazard/ageb/climate_by_ssp_gcm


## 1. File inventory

In [2]:
PATH_RE = re.compile(r"ssp_(?P<ssp>\d+)/gcm_(?P<gcm>[^/]+)/time_period_(?P<time_period>\d{4}_\d{4})/data\.csv$")

files = []
for f in sorted(IN_DIR.rglob("data.csv")):
    m = PATH_RE.search(f.as_posix())
    assert m, f"Unexpected path: {f}"
    files.append({**m.groupdict(), "path": f})
inv = pd.DataFrame(files)
inv["ssp"] = "ssp_" + inv["ssp"]

print(len(inv), "files")
print(inv.groupby(["ssp"]).size().to_dict())
print("GCMs:", inv.gcm.nunique(), "| periodos:", sorted(inv.time_period.unique()))
assert len(inv) == inv.ssp.nunique() * inv.gcm.nunique() * inv.time_period.nunique(), "Missing combinations"
inv.head()

240 files
{'ssp_126': 60, 'ssp_245': 60, 'ssp_370': 60, 'ssp_585': 60}
GCMs: 15 | periodos: ['2021_2040', '2041_2060', '2061_2080', '2081_2100']


,ssp,gcm,time_period,path
0,ssp_126,access_cm2,2021_2040,/Users/fabianfuentes/git/adatpation_ageb_data/...
1,ssp_126,access_cm2,2041_2060,/Users/fabianfuentes/git/adatpation_ageb_data/...
2,ssp_126,access_cm2,2061_2080,/Users/fabianfuentes/git/adatpation_ageb_data/...
3,ssp_126,access_cm2,2081_2100,/Users/fabianfuentes/git/adatpation_ageb_data/...
4,ssp_126,bcc_csm2_mr,2021_2040,/Users/fabianfuentes/git/adatpation_ageb_data/...


## 2. Check that all files have the same columns

Before stacking, the headers of the 240 files are compared.

In [3]:
headers = {r.path: tuple(pd.read_csv(r.path, nrows=0).columns) for r in inv.itertuples()}
ref_cols = next(iter(headers.values()))
bad = {p: c for p, c in headers.items() if c != ref_cols}
print("Reference columns:", ref_cols)
print("Files with a different header:", len(bad))
assert not bad

Reference columns: ('CVEGEO', 'heat_bio2', 'heat_bio5', 'heat_bio10', 'heat_cdd', 'drought_pet_annual', 'drought_aridity_annual', 'cold_bio6', 'cold_bio6_delta', 'flood_bio13', 'flood_bio15')
Files with a different header: 0


## 3. Read and stack

- `CVEGEO` is read as `string` (key with leading zeros and letters, e.g. `010060001009A`).
- `ssp`, `gcm`, `time_period` are added as categorical columns to reduce memory.
- Indicators are stored as `float32` (enough precision for these variables; halves the size).

In [4]:
value_cols = [c for c in ref_cols if c != "CVEGEO"]
dtypes = {"CVEGEO": "string", **{c: "float32" for c in value_cols}}

frames = []
for r in inv.itertuples():
    df = pd.read_csv(r.path, dtype=dtypes)
    df.insert(1, "ssp", r.ssp)
    df.insert(2, "gcm", r.gcm)
    df.insert(3, "time_period", r.time_period)
    frames.append(df)

climate = pd.concat(frames, ignore_index=True)
del frames
for c in ["ssp", "gcm", "time_period"]:
    climate[c] = climate[c].astype("category")
print(climate.shape)
climate.head()

(19548240, 14)


,CVEGEO,ssp,gcm,time_period,heat_bio2,heat_bio5,heat_bio10,heat_cdd,drought_pet_annual,drought_aridity_annual,cold_bio6,cold_bio6_delta,flood_bio13,flood_bio15
0,010060001009A,ssp_126,access_cm2,2021_2040,17.453100,32.0000,22.183300,613.995972,1749.410034,0.284019,3.89257,1.6,137.003006,106.421997
1,0100600010244,ssp_126,access_cm2,2021_2040,17.457800,32.0000,22.200001,615.549988,1749.339966,0.281839,3.80313,1.6,137.000000,106.639999
2,0100600010140,ssp_126,access_cm2,2021_2040,17.443800,32.0065,22.202200,618.651001,1749.819946,0.284638,3.90000,1.6,138.000000,106.666000
3,0100600010278,ssp_126,access_cm2,2021_2040,17.515900,32.0000,22.189400,615.091980,1751.910034,0.282964,3.80000,1.6,137.000000,106.258003
4,0100600010070,ssp_126,access_cm2,2021_2040,17.489799,32.0000,22.186399,613.357971,1750.900024,0.285170,3.80321,1.6,137.772995,106.199997


## 4. Quality checks

In [5]:
n_files = len(inv)
n_ageb = climate.CVEGEO.nunique()
print(f"Unique AGEB: {n_ageb:,} | rows: {len(climate):,} | expected: {n_ageb * n_files:,}")

# 4.1 Each AGEB appears exactly once per combination
key = ["CVEGEO", "ssp", "gcm", "time_period"]
dups = climate.duplicated(key).sum()
print("Duplicates on the key:", dups)
assert dups == 0

# 4.2 All combinations have the same AGEB universe
per_combo = climate.groupby(["ssp", "gcm", "time_period"], observed=True).CVEGEO.nunique()
print("AGEB per combination (min/max):", per_combo.min(), per_combo.max())
assert per_combo.nunique() == 1

# 4.3 CVEGEO length and nulls
print(climate.CVEGEO.str.len().value_counts().to_dict())
print("Nulls per column:")
print(climate.isna().sum()[lambda s: s > 0])

Unique AGEB: 81,451 | rows: 19,548,240 | expected: 19,548,240


Duplicates on the key: 0


AGEB per combination (min/max): 81451 81451
{np.int64(13): 15355680, np.int64(9): 4192560}
Nulls per column:


Series([], dtype: int64)


In [6]:
# 4.4 Summary by time period (national mean, SSP 585 as an example) as a sanity check
(climate[climate.ssp == "ssp_585"]
 .groupby("time_period", observed=True)[value_cols].mean().round(2))

,heat_bio2,heat_bio5,heat_bio10,heat_cdd,drought_pet_annual,drought_aridity_annual,cold_bio6,cold_bio6_delta,flood_bio13,flood_bio15
time_period,,,,,,,,,,
2021_2040,15.24,34.110001,25.650000,1705.050049,1777.489990,0.47,8.91,1.28,181.000000,93.000000
2041_2060,15.39,35.250000,26.719999,1998.150024,1832.030029,0.44,9.73,2.10,177.550003,92.940002
2061_2080,15.50,36.619999,28.030001,2385.929932,1895.430054,0.41,10.83,3.20,174.710007,92.800003
2081_2100,15.64,38.200001,29.490000,2841.699951,1967.130005,0.38,12.04,4.41,170.419998,92.930000


## 5. Export

In [7]:
climate = climate.sort_values(key).reset_index(drop=True)
climate.to_parquet(OUT_PARQUET, index=False, compression="zstd")
print(OUT_PARQUET, f"{OUT_PARQUET.stat().st_size / 1e6:.0f} MB")

# Read-back check: CVEGEO keeps its leading zeros
chk = pd.read_parquet(OUT_PARQUET, columns=["CVEGEO", "ssp", "gcm", "time_period"])
assert len(chk) == len(climate) and chk.CVEGEO.str.len().min() > 0
chk.head()

/Users/fabianfuentes/git/adatpation_ageb_data/data/hazard/ageb/climate_ssp_gcm_all_ageb.parquet 488 MB


,CVEGEO,ssp,gcm,time_period
0,0100100010017,ssp_126,access_cm2,2021_2040
1,0100100010017,ssp_126,access_cm2,2041_2060
2,0100100010017,ssp_126,access_cm2,2061_2080
3,0100100010017,ssp_126,access_cm2,2081_2100
4,0100100010017,ssp_126,bcc_csm2_mr,2021_2040
